# `config` — Framework Configuration

Every knob of the research pipeline lives here so the whole experiment is reproducible from a
single place. Nothing in this notebook downloads market data or does heavy work — it only defines
constants and bundles them into a `config` namespace, so the other notebooks can say
`config.TICKERS`, `config.BETA_LOOKBACK`, and so on.

### Design choices worth flagging
* **Universe: the ~500 S&P 500 constituents**, read from `sp500_universe.csv` (ticker + GICS
  sector). Keeping the list in a versioned file rather than scraping it at run time means a
  re-run months from now uses the *same* universe — scraping fresh would silently change the
  experiment every time index membership shifts.
* **Still a static, present-day list.** This is a broader universe, **not** a point-in-time one:
  it contains today's members, so companies that were dropped from the index never appear.
  **Survivorship bias is reduced in breadth but not eliminated** — `data` shouts about this on
  every load, and it stays in Known Limitations.
* **2018 to 2026.** Spans the 2018 selloff, the COVID crash and recovery, the 2022 bear market
  and the recovery since — several genuinely different regimes rather than one lucky one.
* **Price-based risk factors only** (rolling beta + a dollar-volume size proxy). We deliberately
  avoid fundamentals like book-to-market, whose reporting lags and restatements leak
  future information into a backtest.

In [ ]:
"""Configuration parameters for the Cross-Sectional Alpha Framework."""
import os
import pandas as pd
from types import SimpleNamespace

START_DATE = "2018-01-01"
END_DATE = "2026-08-01"   # pinned (not "today") so re-runs stay reproducible

# --- Universe: the ~500 S&P 500 names, with their GICS sector ---
# Kept in a versioned CSV next to this notebook rather than scraped live: index membership
# changes over time, and a backtest whose universe silently shifts between runs is not an
# experiment, it is a moving target.
_here = os.path.dirname(os.path.abspath("__file__")) if "__file__" not in dir() else ""
_universe = pd.read_csv("sp500_universe.csv")

TICKERS = _universe["ticker"].tolist()
SECTOR_MAPPING = dict(zip(_universe["ticker"], _universe["gics_sector"]))

# Benchmark we regress each stock against to get its market beta.
BENCHMARK = "SPY"

# --- Risk-model hyperparameters ---
BETA_LOOKBACK = 60          # trading days in the rolling beta window (~3 months)
SIZE_PROXY_LOOKBACK = 20    # trading days for the average-volume size proxy (~1 month)
NEUTRALIZATION_MODE = "explicit"  # "explicit" = beta+size+sector, "pca" = statistical factors
PCA_COMPONENTS = 3          # how many statistical factors to strip out in "pca" mode

# --- Evaluation config ---
QUANTILES = 5               # 5 buckets -> a quintile long-short book (~100 names per bucket)
TRANSACTION_COST_BPS = 5.0  # cost in bps charged per unit of ONE-WAY turnover
COST_SENSITIVITY_BPS = [1.0, 5.0, 10.0, 20.0]  # re-price the book at each level
DECAY_HORIZONS = [1, 2, 3, 5, 10]  # forward horizons (days) for the IC-decay curve
TRADING_DAYS_PER_YEAR = 252
ESTIMATED_BREADTH = 120     # assumed independent bets/year for the Fundamental-Law IR proxy

# Diagnostic PNGs land here (path is relative to this notebook's folder).
OUTPUT_DIR = "../output"

# Bundle everything into one object so the other notebooks can just say `config.SOMETHING`,
# the same way the original code did `from alpha_framework import config`.
config = SimpleNamespace(
    START_DATE=START_DATE, END_DATE=END_DATE, TICKERS=TICKERS, BENCHMARK=BENCHMARK,
    SECTOR_MAPPING=SECTOR_MAPPING, BETA_LOOKBACK=BETA_LOOKBACK,
    SIZE_PROXY_LOOKBACK=SIZE_PROXY_LOOKBACK, NEUTRALIZATION_MODE=NEUTRALIZATION_MODE,
    PCA_COMPONENTS=PCA_COMPONENTS, QUANTILES=QUANTILES,
    TRANSACTION_COST_BPS=TRANSACTION_COST_BPS, COST_SENSITIVITY_BPS=COST_SENSITIVITY_BPS,
    DECAY_HORIZONS=DECAY_HORIZONS, TRADING_DAYS_PER_YEAR=TRADING_DAYS_PER_YEAR,
    ESTIMATED_BREADTH=ESTIMATED_BREADTH, OUTPUT_DIR=OUTPUT_DIR,
)

print(f"config loaded: {len(TICKERS)} tickers across {len(set(SECTOR_MAPPING.values()))} GICS "
      f"sectors | {START_DATE} -> {END_DATE} | mode={NEUTRALIZATION_MODE}")